In [1]:
!pip install trl bitsandbytes>=0.46.1

In [2]:
import os
import torch
from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer

dataset_path = "/kaggle/input/datasets/paibarrr/gatra2-chatml-20k/gatra2_chatml_20k.jsonl"
model_id = "aisingapore/Gemma-SEA-LION-v3-9B-IT"
output_dir = "./sea-lion-javanese-gatra-sft"
checkpoint_dir = "/kaggle/input/notebooks/paibarrr/jawani-gatra-2/sea-lion-javanese-gatra-sft/"

dataset = load_dataset("json", data_files=dataset_path, split="train")

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
tokenizer.model_max_length = 512

def format_conversation(example):
    text = tokenizer.apply_chat_template(example["messages"], tokenize=False, add_generation_prompt=False)
    return {"text": text}

dataset = dataset.map(format_conversation, desc="Formatting dataset")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    dtype=torch.bfloat16
)
model.config.use_cache = False
model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model = prepare_model_for_kbit_training(model)

peft_config = LoraConfig(
    r=64,
    lora_alpha=128,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, peft_config)

training_args = TrainingArguments(
    output_dir=output_dir,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    logging_steps=10,
    num_train_epochs=1,
    save_strategy="steps",
    save_steps=250,
    save_total_limit=3,
    optim="paged_adamw_8bit",
    bf16=True,
    max_grad_norm=0.3,
    warmup_steps=50,
    lr_scheduler_type="cosine"
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args
)

# fix: sort by step number, pass path string bukan bool
checkpoint_to_resume = None
if os.path.isdir(checkpoint_dir):
    checkpoints = [d for d in os.listdir(checkpoint_dir) if d.startswith("checkpoint-")]
    if checkpoints:
        latest = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
        checkpoint_to_resume = os.path.join(checkpoint_dir, latest)
        print(f"Resuming from: {checkpoint_to_resume}")
    else:
        print("No checkpoint found, starting fresh.")
else:
    print("Checkpoint dir not found, starting fresh.")

trainer.train(resume_from_checkpoint=checkpoint_to_resume)
trainer.model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)

Generating train split: 0 examples [00:00, ? examples/s]

config.json:   0%|          | 0.00/870 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

Token indices sequence length is longer than the specified maximum sequence length for this model (2699 > 512). Running this sequence through the model will result in indexing errors
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 1, 'pad_token_id': 1}.


Resuming from: /kaggle/input/notebooks/paibarrr/jawani-gatra-2/sea-lion-javanese-gatra-sft/checkpoint-1250


	per_device_train_batch_size: 1 (from args) != 2 (from trainer_state.json)


Step,Training Loss


('./sea-lion-javanese-gatra-sft/tokenizer_config.json',
 './sea-lion-javanese-gatra-sft/chat_template.jinja',
 './sea-lion-javanese-gatra-sft/tokenizer.json')